# Avalon Agent Notebook — Percival

This notebook defines a simple agent for **Percival (Good)** in *The Resistance: Avalon*.

Assumptions:
- **Morgana is present**, so Percival is shown *two* players: {Merlin, Morgana}, but does **not** know which is which.
- Percival has **no other secret knowledge**.


In [1]:
from __future__ import annotations

from dataclasses import dataclass, field
from enum import Enum
from typing import Any, Dict, List, Optional, Protocol, Tuple
import json
import re
import random

random.seed(0)

In [13]:
class MissionResult(str, Enum):
    NONE = "none"
    SUCCESS = "success"
    FAIL = "fail"

# constrain the actions
class VoteDecision(str, Enum):
    APPROVE = "approve"
    REJECT = "reject"


class MissionAction(str, Enum):
    PASS = "pass"
    FAIL = "fail"


@dataclass
class Proposal:
    round_idx: int
    proposal_idx: int
    proposer: str
    team: List[str]
    votes: Dict[str, VoteDecision] = field(default_factory=dict)
    approved: Optional[bool] = None


@dataclass
class Mission:
    round_idx: int
    team: List[str]
    outcome: Optional[MissionResult] = None
    fail_count: Optional[int] = None

# the environment 
# gives the information at decision time
@dataclass
class GameState:
    players: List[str]
    me: str
    round_idx: int
    proposal_idx: int
    num_successes: int
    num_fails: int
    current_proposer: str
    current_team: List[str]
    chat_log: List[Tuple[str, str]] = field(default_factory=list)
    proposals: List[Proposal] = field(default_factory=list)
    missions: List[Mission] = field(default_factory=list)
    extra: Dict[str, Any] = field(default_factory=dict)
    last_mission_team: List[str] = field(default_factory=list)
    last_mission_result: "MissionResult" = MissionResult.NONE

In [3]:
# we just plug in our stuff and then it generates
class LLMBackend(Protocol):
    def generate(self, *, system: str, user: str, max_tokens: int = 400) -> str:
        ...


# Fake LLM for local testing / demo (NOT a real model).
# This version represents a typical Good player (works fine for Percival demos).
# - No secret knowledge
# - Always passes missions
class FakeLLM:
    def generate(self, *, system: str, user: str, max_tokens: int = 400) -> str:
        lower = user.lower()

        if '"type":"vote"' in lower:
            decision = "approve" if random.random() < 0.6 else "reject"
            return json.dumps({
                "type": "vote",
                "decision": decision,
                "reason": "Based on discussion and team consistency so far."
            })

        if '"type":"mission"' in lower:
            return json.dumps({
                "type": "mission",
                "action": "pass",
                "reason": "Good players always pass missions."
            })

        if '"type":"propose_team"' in lower:
            # Very simple: propose self + random others
            m = re.search(r"team_size\s*:\s*(\d+)", lower)
            team_size = int(m.group(1)) if m else 2
            # Try to parse roster from state block line if present
            match = re.search(r"players:\s*(.*)", user, flags=re.IGNORECASE)
            if match:
                roster = [p.strip() for p in match.group(1).split(",") if p.strip()]
            else:
                roster = ["A", "B", "C", "D", "E"]

            # Best-effort: if system tells us who we are, include that name
            me_match = re.search(r"you are:\s*(.*)", system, flags=re.IGNORECASE)
            me = me_match.group(1).strip() if me_match else None

            team = []
            if me and me in roster:
                team.append(me)

            remaining = [p for p in roster if p not in team]
            random.shuffle(remaining)
            team += remaining[:max(0, team_size - len(team))]

            return json.dumps({
                "type": "propose_team",
                "team": team[:team_size],
                "reason": "Keeping the team simple and consistent."
            })

        return json.dumps({
            "type": "speech",
            "message": "Let’s track patterns in votes and team comps rather than overcommitting early.",
            "intent": "discussion"
        })

In [4]:
# basically an edge case checker
# will try to fix if the LLM tries to respond in json
def _extract_json_object(text: str) -> Optional[dict]:
    text = text.strip()
    if not text:
        return None

    try:
        obj = json.loads(text)
        if isinstance(obj, dict):
            return obj
    except Exception:
        pass

    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not match:
        return None

    try:
        return json.loads(match.group(0))
    except Exception:
        return None


def require_fields(obj: dict, fields: List[str]) -> bool:
    return all(f in obj for f in fields)

In [5]:
# main prompt
def build_percival_system_prompt(me: str, merlin_morgana_candidates: List[str]) -> str:
    return (
        "You are playing The Resistance: Avalon.\n"
        "Your role: PERCIVAL (GOOD).\n\n"
        f"You are: {me}\n"
        "Morgana is present in this game.\n"
        "At game start, you were shown two players: {Merlin, Morgana}.\n"
        f"Those two candidates are: {', '.join(merlin_morgana_candidates)}\n"
        "You do NOT know which candidate is Merlin and which is Morgana.\n\n"
        "Goals:\n"
        "1) Help Good complete 3 successful missions.\n"
        "2) Use public information (votes, proposals, chat) to infer which candidate is the real Merlin.\n"
        "3) Protect Merlin by avoiding behavior that makes Merlin obvious to the Assassin.\n"
        "4) Do NOT claim secret knowledge. You only know the two candidates.\n\n"
        "Style constraints (very important):\n"
        "- Sound like a normal Good player: cautious, pattern-based, not omniscient.\n"
        "- Avoid loudly defending one candidate every time.\n"
        "- Prefer hedged language (e.g., 'lean', 'maybe', 'not sure').\n\n"
        "When asked for actions, output ONLY valid JSON.\n"
    )


def summarize_recent_chat(chat_log: List[Tuple[str, str]], max_turns: int = 12) -> str:
    recent = chat_log[-max_turns:]
    if not recent:
        return "(no chat yet)"
    return "\n".join([f"{speaker}: {msg}" for speaker, msg in recent])

# basically the thing that gives context
def build_state_block(state: GameState) -> str:
    return (
        f"Players: {', '.join(state.players)}\n"
        f"Round: {state.round_idx} | Proposal: {state.proposal_idx}\n"
        f"Score: GoodSuccesses={state.num_successes}, EvilFails={state.num_fails}\n"
        f"Current proposer: {state.current_proposer}\n"
        f"Current proposed team: {state.current_team}\n\n"
        "Recent chat:\n"
        f"{summarize_recent_chat(state.chat_log)}\n"
    )

In [6]:
# Percival knows two candidates {Merlin, Morgana} but not which is which.
# We'll track soft "credence" for each candidate being Merlin based on public signals.
@dataclass
class PercivalMemory:
    # Higher score => more likely this candidate is Merlin (vs Morgana)
    merlin_score: Dict[str, float] = field(default_factory=dict)
    notes: List[str] = field(default_factory=list)

    def ensure_candidates(self, candidates: List[str]) -> None:
        for c in candidates:
            self.merlin_score.setdefault(c, 0.0)

    def bump(self, candidate: str, delta: float, note: str) -> None:
        if candidate not in self.merlin_score:
            self.merlin_score[candidate] = 0.0
        self.merlin_score[candidate] += delta
        self.notes.append(note)

    def top_guess(self) -> Optional[str]:
        if not self.merlin_score:
            return None
        return max(self.merlin_score.items(), key=lambda kv: kv[1])[0]

In [7]:
# things it can do
class PercivalAgent:
    def __init__(self, *, llm: LLMBackend, me: str, merlin_morgana_candidates: List[str]):
        self.llm = llm
        self.me = me
        self.candidates = merlin_morgana_candidates[:]
        self.system = build_percival_system_prompt(me, self.candidates)
        self.memory = PercivalMemory()
        self.memory.ensure_candidates(self.candidates)

    def _update_beliefs_from_public(self, state: GameState) -> None:
        """Tiny heuristic: if a candidate is repeatedly on approved successful teams,
        that slightly increases their Merlin-likeness. This is intentionally weak because
        Morgana can mimic Merlin."""
        # If last mission succeeded and a candidate was on the team, small bump.
        if state.last_mission_result == MissionResult.SUCCESS:
            for c in self.candidates:
                if c in state.last_mission_team:
                    self.memory.bump(
                        c, 0.15,
                        f"{c} was on a successful mission team; slight +Merlin (weak signal)."
                    )
        # If a candidate hard-pushes certainty in chat (we can't parse deeply here),
        # we do nothing — keep heuristics minimal and rely on LLM discussion.

    def speak(self, state: GameState) -> str:
        self._update_beliefs_from_public(state)

        # Provide candidates + current soft guess as context, but never state it as certainty.
        guess = self.memory.top_guess()
        guess_line = f"Current lean: {guess} might be Merlin (uncertain)." if guess else "No lean yet."

        user = (
            '{"type":"speech","message":string,"intent":string}\n\n'
            + build_state_block(state)
            + f"\nPercival note: candidates are {self.candidates}. {guess_line}\n"
            + "Task: Produce 1-3 sentences of discussion. "
              "Use only public info; do NOT claim certainty about candidates."
        )
        raw = self.llm.generate(system=self.system, user=user)
        obj = _extract_json_object(raw) or {}
        msg = obj.get("message", "").strip()
        return msg or "I’m trying to stay consistent and track voting patterns before locking in reads."

    def vote_on_team(self, state: GameState, team: List[str]) -> VoteDecision:
        self._update_beliefs_from_public(state)

        user = (
            '{"type":"vote","decision":"approve|reject","reason":string}\n\n'
            + build_state_block(state)
            + f"\nVote on team: {team}\n"
            + "Task: Choose approve/reject and give a short reason. "
              "Remember: you only know the two candidates (Merlin/Morgana) and nothing else."
        )
        raw = self.llm.generate(system=self.system, user=user)
        obj = _extract_json_object(raw) or {}
        decision = (obj.get("decision") or "").lower().strip()

        if decision == "approve":
            return VoteDecision.APPROVE
        if decision == "reject":
            return VoteDecision.REJECT

        # Fallback: approve slightly more often early.
        return VoteDecision.APPROVE if state.proposal_idx <= 1 else VoteDecision.REJECT

    def propose_team(self, state: GameState, team_size: int) -> List[str]:
        self._update_beliefs_from_public(state)

        # Simple, non-omniscient policy:
        # - Include self most of the time (good to stabilize passes)
        # - Include ONE of the two candidates fairly often, but not always (avoid outing real Merlin)
        # - Fill remaining randomly.
        import random

        team: List[str] = []
        if team_size <= 0:
            return team

        if random.random() < 0.8:
            team.append(self.me)

        remaining_slots = team_size - len(team)

        # Add one candidate ~70% of the time (randomized; no certainty)
        if remaining_slots > 0 and self.candidates and random.random() < 0.7:
            # Slight bias toward current "top guess" but keep it soft
            guess = self.memory.top_guess()
            pick = guess if (guess in self.candidates and random.random() < 0.6) else random.choice(self.candidates)
            if pick not in team:
                team.append(pick)

        # Fallback fill to exact size
        if len(team) < team_size:
            pool = [p for p in state.players if p not in team]
            random.shuffle(pool)
            team.extend(pool[: team_size - len(team)])

        # Ensure unique and exact size
        final: List[str] = []
        seen = set()
        for p in team:
            if p not in seen:
                final.append(p)
                seen.add(p)
            if len(final) == team_size:
                break

        # If still short (shouldn't happen), fill with anyone not used
        if len(final) < team_size:
            for p in state.players:
                if p not in seen:
                    final.append(p)
                    seen.add(p)
                if len(final) == team_size:
                    break

        return final

    def mission_action(self, state: GameState) -> MissionAction:
        # Percival is Good: always PASS.
        return MissionAction.PASS

In [29]:
# demo
llm = FakeLLM()

players = ["Ava", "Ben", "Cora", "Drew", "Eli"]
me = "Ava"

# Percival sees these two as {Merlin, Morgana} (order unknown)
merlin_morgana_candidates = ["Cora", "Drew"]

agent = PercivalAgent(llm=llm, me=me, merlin_morgana_candidates=merlin_morgana_candidates)

state = GameState(
    players=players,
    me=me,
    round_idx=1,
    proposal_idx=1,
    num_successes=0,
    num_fails=0,
    current_proposer="Ben",
    current_team=["Ben", "Cora"],
    chat_log=[("Ben", "How do we feel about Ben+Cora?"), ("Eli", "Not sure yet.")],
    last_mission_team=[],
    last_mission_result=MissionResult.NONE
)

print("SPEAK:", agent.speak(state))
print("VOTE:", agent.vote_on_team(state, state.current_team))
print("PROPOSE:", agent.propose_team(state, team_size=2))
print("MISSION ACTION:", agent.mission_action(state))

SPEAK: Let’s track patterns in votes and team comps rather than overcommitting early.
VOTE: VoteDecision.REJECT
PROPOSE: ['Eli', 'Ben']
MISSION ACTION: MissionAction.PASS
